# 노트북 5. 생성 파라미터

> Phase 2 — 제어: 모델 동작을 원하는 대로 다루기

같은 모델, 같은 프롬프트인데 결과가 매번 다릅니다.
이유를 이해하고, 용도에 맞게 제어할 수 있어야 합니다.

**학습 목표**
- Temperature가 토큰 선택 확률 분포에 미치는 영향을 이해한다
- Top-p, Top-k의 역할과 Temperature와의 상호작용을 설명할 수 있다
- max_output_tokens, stop_sequences, seed를 활용하여 출력을 제어할 수 있다
- 용도별 최적 파라미터 조합을 스스로 도출할 수 있다
- LangChain에서 동일한 파라미터를 적용하는 방법을 안다

**구성**
| Part | 내용 | 형태 |
|------|------|------|
| Part 1 — 이론 | Temperature + Top-p + Top-k + 기타 파라미터 + LangChain 연동 | 읽고 실행 |
| Part 2 — 실습 | 파라미터별 실험 + 조합 매트릭스 + 실전 적용 | 직접 작성 |
| Part 3 — 챌린지 | 용도별 최적 파라미터 가이드 도출 | 강사와 함께 |

In [ ]:
# 환경 설정 — 이 셀을 먼저 실행하세요
!pip install -q google-genai langchain-google-genai

import os
from google import genai

# API 키 설정 (Colab 환경 기준)
# 왼쪽 키 아이콘 → GEMINI_API_KEY 등록 후 실행
from google.colab import userdata
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=GEMINI_API_KEY)

MODEL = "gemini-2.5-flash"
print("환경 설정 완료")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 1.1 MB/s eta 0:00:00
환경 설정 완료


---

# Part 1 — 이론

개념을 마크다운 설명과 실행 가능한 코드 예시로 배웁니다.
읽고 실행하면서 따라와 주세요.

## 1.1 왜 같은 질문에 다른 결과가 나오는가?

LLM은 다음 토큰을 **확률적으로** 선택합니다.
"오늘 날씨가" 다음에 올 수 있는 토큰이 여러 개 있고, 모델은 그 중 하나를 확률에 따라 "뽑습니다".

```
"오늘 날씨가" → 좋습니다(40%)  맑습니다(25%)  흐립니다(15%)  ...
```

이 확률 분포를 어떻게 조절하느냐에 따라 모델의 출력이 달라집니다.
이것을 조절하는 도구가 **생성 파라미터(Generation Parameters)**입니다.

| 파라미터 | 역할 | 주요 값 범위 |
|---------|------|------------|
| temperature | 확률 분포의 뾰족함 | 0.0 ~ 2.0 |
| top_p | 누적 확률 기반 필터링 | 0.0 ~ 1.0 |
| top_k | 상위 k개 후보만 남김 | 1 ~ 100+ |
| max_output_tokens | 출력 길이 상한 | 1 ~ 65,536 |
| stop_sequences | 생성 중단 트리거 | 문자열 리스트 |
| seed | 재현성을 위한 시드 | 정수 |

## 1.2 Temperature

**Temperature**(온도 파라미터)는 토큰 선택 확률 분포의 "뾰족함"을 조절합니다.

- **temperature = 0**: 가장 확률이 높은 토큰만 선택 (거의 결정적)
- **temperature = 1.0**: 원래 학습된 확률 분포 그대로
- **temperature > 1.0**: 분포가 평탄해져 낮은 확률의 토큰도 선택될 수 있음

> 핵심: temperature는 "창의성 다이얼"이라고 생각하면 됩니다.
> 낮으면 정확하고 일관되지만 뻔한 답, 높으면 다양하지만 엉뚱한 답.

아래 코드는 temperature 0.0과 1.5에서 동일한 프롬프트의 결과가 어떻게 달라지는지 보여줍니다.

In [ ]:
# temperature에 따른 응답 차이 비교
prompt = "'어린 시절'하면 떠오르는 단어 5개를 나열해줘."

# temperature=0.0: 거의 결정적 — 매번 비슷한 답
for i in range(1,3):
    response_low = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config={"temperature": 0.0, "thinking_config": {"thinking_budget": 0}},
    )
    print(f"[temperature=0.0] {i}번째\n{response_low.text}")

print("---")

for i in range(1,3):
    # temperature=1.5: 높은 무작위성 — 매번 다른 답
    response_high = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config={"temperature": 1.5, "thinking_config": {"thinking_budget": 0}},
    )
    print(f"[temperature=1.5] {i}번째\n{response_high.text}")

[temperature=0.0] 1번째
'어린 시절'하면 떠오르는 단어 5개는 다음과 같습니다.

1. **놀이터**
2. **친구**
3. **장난감**
4. **엄마**
5. **학교**
[temperature=0.0] 2번째
'어린 시절'하면 떠오르는 단어 5개는 다음과 같습니다.

1. **놀이터**
2. **친구**
3. **장난감**
4. **엄마**
5. **학교**
---
[temperature=1.5] 1번째
'어린 시절'하면 떠오르는 단어 5개는 다음과 같습니다:

1. **놀이**
2. **친구**
3. **가족**
4. **학교**
5. **동화**
[temperature=1.5] 2번째
'어린 시절'하면 떠오르는 단어 5개는 다음과 같습니다:

1.  **놀이터**
2.  **친구**
3.  **웃음**
4.  **꿈**
5.  **동화**


### temperature = 0의 결정성

temperature를 0으로 설정하면 항상 가장 확률 높은 토큰을 선택합니다.
이론적으로는 매번 동일한 결과가 나와야 하지만, 실제로는 서버 내부의 병렬 처리 등으로 미세한 차이가 발생할 수 있습니다.

아래 코드는 temperature=0으로 같은 질문을 3번 호출하여 결과를 비교합니다.

In [ ]:
# temperature=0 반복 호출 — 결정성 확인
prompt = "대한민국의 수도는?"

results = []
for i in range(3):
    resp = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config={"temperature": 0.0, "thinking_config": {"thinking_budget": 0}},
    )
    results.append(resp.text.strip())
    print(f"호출 {i+1}: {results[-1]}")

# 결과 일치 확인
all_same = len(set(results)) == 1
print(f"\n모든 결과 동일: {all_same}")

호출 1: 대한민국의 수도는 **서울**입니다.
호출 2: 대한민국의 수도는 **서울**입니다.
호출 3: 대한민국의 수도는 **서울**입니다.

모든 결과 동일: True


### Temperature 스케일 비교

temperature를 세밀하게 조절하면 다양성을 미세 조정할 수 있습니다.
일반적인 사용 범위는 다음과 같습니다.

| temperature | 용도 | 특징 |
|------------|------|------|
| 0.0 | 분류, 추출, 코드 생성 | 가장 확률 높은 답만 선택 |
| 0.3 | 요약, Q&A | 약간의 변화 허용 |
| 0.7 | 일반 대화 | 자연스러운 다양성 |
| 1.0 | 창작, 브레인스토밍 | 학습된 분포 그대로 |
| 1.5+ | 실험적 | 매우 무작위적, 비문 가능 |

아래 코드는 5단계의 temperature에서 동일 프롬프트의 결과를 비교합니다.

In [ ]:
# 5단계 temperature 비교
prompt = "커피를 마시면 좋은 점을 한 문장으로 알려줘"
temperatures = [0.0, 0.3, 0.7, 1.0, 1.5]

for temp in temperatures:
    resp = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config={"temperature": temp, "thinking_config": {"thinking_budget": 0}},
    )
    print(f"[temp={temp}] {resp.text.strip()[:100]}")

[temp=0.0] 커피를 마시면 집중력 향상, 피로 감소, 기분 전환 등 다양한 긍정적인 효과를 얻을 수 있습니다.
[temp=0.3] 커피를 마시면 집중력 향상과 활력 증진에 도움을 받을 수 있습니다.
[temp=0.7] 커피를 마시면 정신이 맑아지고 집중력이 향상되어 업무나 학업 효율을 높일 수 있습니다.
[temp=1.0] 커피를 마시면 정신이 맑아지고 집중력이 향상되어 일상 활동에 활력을 더할 수 있습니다.
[temp=1.5] 커피는 기분을 좋게 하고 집중력을 높여주는 각성제 역할을 합니다.


### 높은 Temperature와 창의적 작업

창의적 작업(소설, 시, 아이디어 발상)에서는 높은 temperature가 유용합니다.
하지만 너무 높으면 문법이 무너지거나 의미 없는 텍스트가 생성될 수 있습니다.

아래 코드는 창의적 프롬프트에서 temperature의 효과를 보여줍니다.

In [ ]:
# 창의적 프롬프트에서 temperature 효과
creative_prompt = "'빗소리'라는 단어로 시작하는 짧은 시를 써줘. 2줄로."

for temp in [0.0, 0.7, 1.5]:
    resp = client.models.generate_content(
        model=MODEL,
        contents=creative_prompt,
        config={"temperature": temp, "thinking_config": {"thinking_budget": 0}},
    )
    print(f"--- temperature={temp} ---")
    print(resp.text.strip())
    print()

--- temperature=0.0 ---
빗소리, 창밖을 두드리는 리듬.
내 마음도 촉촉이 젖어드네.

--- temperature=0.7 ---
빗소리, 창문을 두드리는 리듬.
내 마음도 따라 춤추네, 촉촉한 멜로디.

--- temperature=1.5 ---
빗소리, 창문을 두드리는 나른한 박자
땅은 목마름을 잊고 축복에 잠긴다.



> 정리: temperature는 가장 자주 조절하는 파라미터입니다.
> 정확성이 중요한 작업에는 0.0 ~ 0.3, 자연스러운 대화에는 0.5 ~ 0.7, 창의적 작업에는 0.8 ~ 1.0가 일반적입니다.

## 1.3 Top-p (Nucleus Sampling)

**Top-p**(또는 Nucleus Sampling)는 누적 확률이 p에 도달할 때까지의 토큰만 후보로 남기는 필터입니다.

예시: `top_p = 0.9`일 때
```
좋습니다(40%) + 맑습니다(25%) + 흐립니다(15%) + 따뜻합니다(10%) = 90%
→ 이 4개만 후보로 남기고, 나머지 토큰은 제외
```

- **top_p = 1.0**: 필터링 없음 (기본값, 모든 토큰이 후보)
- **top_p = 0.9**: 상위 90% 확률의 토큰만 후보
- **top_p = 0.5**: 상위 50% 확률의 토큰만 후보 (매우 제한적)

> 핵심: Top-p는 "확률이 너무 낮은 엉뚱한 토큰을 걸러내는 안전망"입니다.

아래 코드는 top_p 값에 따른 응답 차이를 비교합니다.

In [ ]:
# top_p에 따른 응답 비교
prompt = "봄에 하면 좋은 활동 3가지를 추천해줘"
top_p_values = [1.0, 0.9, 0.5, 0.1]

for top_p in top_p_values:
    resp = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config={
            "temperature": 1.0,  # temperature를 고정하고 top_p만 변경
            "top_p": top_p,
            "thinking_config": {"thinking_budget": 0},
        },
    )
    # 첫 줄만 출력
    first_line = resp.text.strip().split("\n")[0][:80]
    print(f"[top_p={top_p}] {first_line}")

[top_p=1.0] 따뜻한 봄날을 만끽할 수 있는 특별한 활동 3가지를 추천해 드릴게요! 🌸
[top_p=0.9] 봄은 만물이 소생하고 따뜻한 햇살이 가득한 계절이라 야외 활동하기에 정말 좋아요. 봄에 하면 좋은 활동 3가지를 추천해 드릴게요!
[top_p=0.5] 따뜻한 봄날, 활기찬 기운을 만끽할 수 있는 활동 3가지를 추천해 드릴게요!
[top_p=0.1] 따뜻한 봄날, 몸과 마음을 충전할 수 있는 활동 3가지를 추천해 드릴게요!


### Temperature + Top-p 상호작용

Temperature와 Top-p는 함께 작용합니다.
Temperature가 확률 분포를 변형한 뒤, Top-p가 후보를 필터링합니다.

```
원래 분포 → Temperature 적용 → Top-p 필터링 → 최종 토큰 선택
```

둘 다 높이면 매우 무작위적이고, 둘 다 낮추면 매우 결정적입니다.

| 조합 | 특징 |
|------|------|
| temp=0, top_p=1.0 | 완전 결정적 |
| temp=0.7, top_p=0.9 | 일반적 추천 조합 |
| temp=1.5, top_p=0.95 | 높은 다양성, 안전망 있음 |
| temp=1.5, top_p=1.0 | 최대 무작위 (비문 위험) |

아래 코드는 4가지 조합의 결과를 비교합니다.

In [ ]:
# Temperature + Top-p 조합 비교
prompt = "다음의 표현을 쓸 법한 시의 제목을 알려주세요, '그 순간에 날개 없는 벌레들이 일제히 허공으로 날아올랐다.'"

combos = [
    (0.0, 1.0, "결정적"),
    (0.7, 0.9, "일반 추천"),
    (1.5, 0.95, "높은 다양성"),
    (1.5, 1.0, "최대 무작위"),
]

for _ in range(3):
    for temp, top_p, label in combos:
        resp = client.models.generate_content(
            model=MODEL,
            contents=prompt,
            config={
                "temperature": temp,
                "top_p": top_p,
                "max_output_tokens": 128,
                "thinking_config": {"thinking_budget": 0},
            },
        )
        text = resp.text.strip()
        print(f"[{label}: temp={temp}, top_p={top_p}]")
        print(f"  {text}\n")
    print("="*50)

[결정적: temp=0.0, top_p=1.0]
  '그 순간에 날개 없는 벌레들이 일제히 허공으로 날아올랐다.'라는 표현을 쓸 법한 시의 제목은 다음과 같습니다.

**직접적인 제목:**

* **날개 없는 비상**
* **허공의 군무**
* **벌레들의 환희**
* **순간의 기적**
* **날아오르다, 벌레들**

**은유적이고 상징적인 제목:**

* **중력의 역설**
* **존재의 도약**
* **미미한 자들의

[일반 추천: temp=0.7, top_p=0.9]
  '그 순간에 날개 없는 벌레들이 일제히 허공으로 날아올랐다.'와 같은 표현을 쓸 법한 시의 제목은 다음과 같습니다.

*   **변모의 순간:** 벌레들이 날아오르는 행위 자체가 어떤 변화나 기적적인 변모를 상징하기 때문입니다.
*   **날개 없는 비상(飛上):** 역설적인 표현으로, 날개가 없음에도 불구하고 날아오르는 행위의 특별함을 강조합니다.
*   **중력의 배신:** 보통 중력에 묶여 있는 벌레들이

[높은 다양성: temp=1.5, top_p=0.95]
  '그 순간에 날개 없는 벌레들이 일제히 허공으로 날아올랐다.'와 같은 표현이 쓰일 법한 시의 제목은 다음과 같습니다.

*   **변성(變星)의 전조:** 별이 변하는 것의 전조처럼, 갑작스럽고 기이한 변화를 묘사하는 데 어울립니다.
*   **어둠 속의 환희:** 기이하고 부자연스러운 현상에서 오히려 기이한 환희나 에너지를 발견하는 느낌을 줍니다.
*   **역류(逆流)하는

[최대 무작위: temp=1.5, top_p=1.0]
  이처럼 서정적이면서도 약동적인 초현실적인 표현을 담을 수 있는 시의 제목들을 적어드릴게요. 표현의 분위기와 의도를 여러 각도로 해석하여 적어보았습니다.

**움직임, 전환의 강조:**
* **벌레들의 비상(飛上)**
* **탈피(脫皮)의 순간**
* **꿈틀거림의 날개**
* **허공으로의 도약**
* **일제히 꿈꾸다(or 꿈꾸다 잠깨다)**
* **날개 없는 나는 것들**

[결정적: t

> 실무 가이드라인: 대부분의 경우 temperature만 조절하면 충분합니다.
> top_p는 0.9~0.95로 고정하고, temperature로 다양성을 조절하는 것이 가장 일반적인 패턴입니다.
> 둘 다 극단적으로 설정하는 것은 피하세요.

## 1.4 Top-k

**Top-k**는 확률 상위 k개 토큰만 후보로 남기는 필터입니다.
Top-p가 확률 비율로 필터링하는 반면, Top-k는 개수로 필터링합니다.

- **top_k = 1**: 가장 확률 높은 토큰 1개만 (= temperature 0과 유사)
- **top_k = 40**: 상위 40개 토큰 중에서 선택 (Gemini 기본값)
- **top_k = 100+**: 거의 필터링 없음

> 참고: Top-k는 Gemini에서는 지원하지만 OpenAI API에는 없는 파라미터입니다.
> 모델마다 지원하는 파라미터가 다르므로, 문서를 확인하는 습관이 중요합니다.

아래 코드는 top_k 값에 따른 응답 차이를 비교합니다.

In [ ]:
# top_k에 따른 응답 비교
prompt = "재미있는 파이썬 프로젝트 아이디어를 하나 제안해줘"
top_k_values = [1, 1, 1, 1]

for top_k in top_k_values:
    resp = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config={
            "temperature": 1.0,
            "top_k": top_k,
            "thinking_config": {"thinking_budget": 0},
        },
    )
    text = resp.text.strip()
    print(f"[top_k={top_k:>3}] {text}")

[top_k=  1] 재미있는 파이썬 프로젝트 아이디어는 **이미지 스티커/밈 생성기**입니다!

이 프로젝트는 파이썬의 이미지 처리 라이브러리(PIL/Pillow)를 활용하여 사용자에게 다음과 같은 기능을 제공합니다.

**핵심 기능:**

1.  **이미지 업로드/선택:** 사용자가 자신의 로컬 이미지를 업로드하거나, 미리 정의된 배경 이미지를 선택할 수 있습니다.
2.  **텍스트 오버레이:**
    *   사용자가 원하는 텍스트를 입력합니다.
    *   텍스트의 글꼴, 크기, 색상, 윤곽선(border/stroke)을 조절할 수 있습니다.
    *   텍스트 위치를 마우스 드래그로 조절하거나, 상하좌우 정렬 옵션을 제공합니다.
3.  **이미지 오버레이 (스티커):**
    *   미리 준비된 PNG 스티커 이미지(투명 배경) 목록을 제공합니다. (예: 안경, 모자, 말풍선, 인기 밈 캐릭터 등)
    *   사용자가 원하는 스티커를 선택하여 이미지에 추가합니다.
    *   스티커의 크기, 회전, 위치를 조절할 수 있습니다.
4.  **필터/효과 (선택 사항):**
    *   간단한 이미지 필터(흑백, 세피아, 블러 등)를 적용할 수 있는 기능을 추가합니다.
5.  **결과 저장:** 생성된 최종 이미지를 JPG 또는 PNG 파일로 저장합니다.

**왜 이 프로젝트가 재미있을까요?**

*   **시각적인 결과:** 바로 눈으로 확인할 수 있는 결과물이 나와 성취감이 높습니다.
*   **창의성 발휘:** 사용자가 자신만의 밈이나 재미있는 스티커 이미지를 만들 수 있어 창의력을 자극합니다.
*   **실용적인 활용:** 친구들에게 보낼 재미있는 이미지를 만들거나, 소셜 미디어 콘텐츠를 만드는 데 활용될 수 있습니다.
*   **다양한 파이썬 기술 학습:**
    *   **GUI 프로그래밍:** `Tkinter`, `PyQt`, `Kivy` 등 GUI 라이브러리를 사용하여 사용자 인터페이스를 만듭니다. (Tkinter가 가장 접근하기

> 참고: Top-k와 Top-p의 차이
> - Top-k: 항상 정확히 k개의 후보. 확률 분포의 모양과 무관
> - Top-p: 확률 분포에 따라 후보 수가 동적으로 변함. 분포가 뾰족하면 적은 수, 평탄하면 많은 수
> - 실무에서는 Top-p가 더 많이 사용됩니다. 분포 적응적이기 때문입니다.

## 1.5 파라미터 조합 전략

Temperature, Top-p, Top-k는 독립적으로 사용할 수도 있지만, 함께 조합하는 것이 일반적입니다.
아래는 용도별 추천 프리셋입니다.

| 용도 | temperature | top_p | top_k | 설명 |
|------|-----------|-------|-------|------|
| 코드 생성 | 0.0 | 1.0 | - | 정확성 최우선 |
| 분류/추출 | 0.0~0.2 | 1.0 | - | 일관된 결과 |
| 요약/Q&A | 0.3 | 0.95 | 40 | 약간의 변화 허용 |
| 일반 대화 | 0.7 | 0.9 | 40 | 자연스러운 다양성 |
| 창작/브레인스토밍 | 1.0~1.2 | 0.95 | 60 | 높은 다양성 |

아래 코드는 프리셋별 결과를 비교합니다.

In [ ]:
# 용도별 프리셋 비교
presets = {
    "코드 생성": {"temperature": 0.0, "top_p": 1.0},
    "요약/Q&A": {"temperature": 0.3, "top_p": 0.95, "top_k": 40},
    "일반 대화": {"temperature": 0.7, "top_p": 0.9, "top_k": 40},
    "창작": {"temperature": 1.0, "top_p": 0.95, "top_k": 60},
}

prompt = "'시간'이라는 주제로 한 문장을 써줘"

for name, params in presets.items():
    config = {**params, "thinking_config": {"thinking_budget": 0}}
    resp = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=config,
    )
    print(f"[{name}] {resp.text.strip()[:100]}")

[코드 생성] 시간은 덧없이 흐르지만, 그 속에서 우리는 영원을 꿈꾼다.
[요약/Q&A] 시간은 덧없이 흐르지만, 그 속에서 우리는 영원을 꿈꾼다.
[일반 대화] "시간은 멈추지 않고 흐르지만, 그 속에서 우리는 매 순간 새로운 의미를 찾아낼 수 있다."
[창작] 시간은 덧없이 흐르는 강물과 같아서, 붙잡으려 해도 어느새 저 멀리 흘러가 버린다.


### 매트릭스 실험

Temperature와 Top-p의 조합을 체계적으로 실험해봅시다.
표 형태로 결과를 정리하면 패턴을 쉽게 파악할 수 있습니다.

아래 코드는 3x3 매트릭스 실험을 수행합니다.

매트릭스 결과를 관찰하면 몇 가지 패턴이 보입니다.

- **temp=0.0 행**: top_p를 바꿔도 결과가 거의 동일합니다. temperature가 0이면 이미 하나의 토큰만 선택하므로 top_p 필터링이 무의미합니다.
- **temp=1.5 행**: top_p에 따라 결과가 크게 달라집니다. 높은 temperature로 분포가 평탄해진 상태에서 top_p가 안전망 역할을 합니다.
- **대각선**: temp=0/top_p=1.0(가장 결정적) → temp=1.5/top_p=1.0(가장 무작위)으로 갈수록 다양성이 증가합니다.

In [ ]:
# Temperature x Top-p 매트릭스 실험
from itertools import product

prompt = "삶과 죽음의 차이에 대해 설명해줘"
temps = [0.0, 0.5, 1.0]
top_ps = [0.2, 0.6, 1.0]

for temp, tp in product(temps, top_ps):
    resp = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config={
            "temperature": temp,
            "top_p": tp,
            "thinking_config": {"thinking_budget": 0},
        },
    )
    text = resp.text.strip().replace("\n", " ")
    print(f"temperature {temp}, top-p {tp}, {text}", end="")
    print()

temperature 0.0, top-p 0.2, 삶과 죽음은 존재의 양극단에 있는 개념으로, 여러 면에서 극명한 차이를 보입니다.  ### 삶 (Life)  삶은 유기체가 태어나서 죽기 전까지의 모든 과정을 의미하며, 다음과 같은 특징을 가집니다.  *   **성장과 발달:** 유기체는 태어나면서부터 성장하고 발달하며, 이는 물리적인 크기 증가뿐 아니라 복잡성 증가, 기능 향상 등을 포함합니다. *   **대사 활동:** 생명체는 에너지를 얻고 사용하기 위해 끊임없이 물질을 분해하고 합성하는 대사 활동을 합니다. 호흡, 소화, 순환 등이 이에 해당합니다. *   **자극에 대한 반응:** 외부 환경의 변화나 자극에 대해 반응하고 적응합니다. 예를 들어, 추우면 몸을 웅크리거나 따뜻한 곳을 찾습니다. *   **생식:** 자신의 유전 정보를 다음 세대로 전달하여 종족을 보존합니다. *   **항상성 유지:** 내부 환경을 일정하게 유지하려는 경향이 있습니다. 체온 조절, 혈당 조절 등이 대표적입니다. *   **의식과 감정 (고등 생명체):** 특히 인간과 같은 고등 생명체는 생각하고 느끼며, 자아를 인식하는 의식과 다양한 감정을 가집니다. *   **목표와 의미 부여:** 삶의 목적을 설정하고, 경험을 통해 의미를 부여하며, 미래를 계획합니다. *   **변화와 적응:** 끊임없이 변화하는 환경에 적응하며 진화합니다.  ### 죽음 (Death)  죽음은 생명 활동이 영구적으로 정지하는 상태를 의미하며, 다음과 같은 특징을 가집니다.  *   **생명 활동의 영구적 정지:** 심장 박동, 호흡, 뇌 활동 등 모든 생명 유지 기능이 멈춥니다. *   **대사 활동 중단:** 에너지를 생산하고 사용하는 모든 대사 활동이 중단됩니다. *   **자극에 대한 무반응:** 외부 자극에 전혀 반응하지 않습니다. *   **성장과 발달의 종료:** 더 이상 성장하거나 발달하지 않습니다. *   **항상성 상실:** 내부 환경을 일정하게 유지하려는 노력이 중단되고, 외부 

KeyboardInterrupt: 

In [ ]:
# Temperature x Top-p 매트릭스 실험
from itertools import product

prompt = "삶과 죽음의 차이에 대해 설명해줘"
temps = [0.0, 0.5, 1.0]
top_ps = [0.2, 0.6, 1.0]

for temp, tp in product(temps, top_ps):
    resp = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config={
            "temperature": temp,
            "top_p": tp,
            "thinking_config": {"thinking_budget": 1024},
        },
    )
    text = resp.text.strip().replace("\n", " ")
    print(f"temperature {temp}, top-p {tp}, {text}", end="")
    print()

temperature 0.0, top-p 0.2, 삶과 죽음은 존재의 가장 근본적인 두 가지 상태이며, 겉으로는 극명하게 대립하지만 실제로는 서로를 정의하고 의미를 부여하는 관계에 있습니다. 이 둘의 차이를 여러 관점에서 설명해 드릴게요.  ---  ### 삶과 죽음의 차이  **1. 생물학적/생리적 차이**  *   **삶 (Life):**     *   **생명 활동:** 신진대사(영양분 섭취, 에너지 생산, 노폐물 배출), 호흡, 순환, 세포 분열 및 성장, 항상성 유지(체온, 혈당 등 내부 환경 조절)와 같은 모든 생명 활동이 활발하게 일어납니다.     *   **조직화:** 세포, 조직, 기관, 기관계가 유기적으로 연결되어 복잡한 기능을 수행합니다.     *   **정보 처리:** 유전 정보(DNA)에 따라 생명 활동이 이루어지고, 외부 자극에 반응하며 정보를 처리합니다. *   **죽음 (Death):**     *   **생명 활동의 정지:** 모든 신진대사, 호흡, 순환, 뇌 활동 등 생명 유지에 필수적인 기능이 영구적으로 멈춥니다.     *   **조직화의 붕괴:** 세포와 조직의 기능이 정지하고, 시간이 지남에 따라 분해(부패)가 시작됩니다.     *   **정보 처리의 중단:** 뇌 활동이 멈추면서 외부 자극에 대한 반응이나 정보 처리가 완전히 중단됩니다.  **2. 기능적/행동적 차이**  *   **삶 (Life):**     *   **성장과 발달:** 유기체는 태어나서 성장하고 발달하며 변화합니다.     *   **번식:** 자신의 유전자를 다음 세대에 전달하여 종족을 유지합니다.     *   **자극 반응:** 외부 환경의 변화나 자극에 대해 능동적으로 반응하고 적응합니다.     *   **움직임:** 스스로 움직이거나 내부적으로 활동합니다. *   **죽음 (Death):**     *   **성장과 발달의 중단:** 더 이상 성장하거나 발달하지 않습니다.     *   **번식 능력 상실:** 번식 능력을 완전히 상

## 1.6 max_output_tokens

**max_output_tokens**는 모델이 생성하는 출력의 최대 토큰 수를 제한합니다.

- 설정하지 않으면 모델 기본 최대값(gemini-2.5-flash: 65,536) 적용
- 짧은 답변이 필요할 때 비용과 속도를 절약할 수 있음
- 토큰 한도에 도달하면 문장 중간에서 잘릴 수 있음

아래 코드는 max_output_tokens에 따른 출력 길이 차이를 보여줍니다.

In [ ]:
# max_output_tokens에 따른 출력 길이
prompt = "파이썬 프로그래밍의 장점을 설명해주세요."
max_tokens_list = [20, 50, 200]

for max_tokens in max_tokens_list:
    resp = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config={
            "max_output_tokens": max_tokens,
            "thinking_config": {"thinking_budget": 0},
        },
    )
    usage = resp.usage_metadata
    print(f"[max={max_tokens:>3}] 실제 출력: {usage.candidates_token_count}토큰")
    print(f"  {resp.text.strip()[:120]}")
    print()

[max= 20] 실제 출력: 20토큰
  파이썬 프로그래밍은 다양한 장점을 가지고 있어 많은 개발자들에게 사랑받고 있습니다

[max= 50] 실제 출력: 50토큰
  ## 파이썬 프로그래밍의 장점

파이썬은 배우기 쉽고, 강력하며, 다양한 분야에서 활용될 수 있는 다재다능한 프로그래밍 언어입니다. 이러한 특징들 덕분에 파이썬

[max=200] 실제 출력: 200토큰
  파이썬은 배우기 쉽고, 다양한 분야에 활용될 수 있으며, 효율적인 프로그래밍을 가능하게 하는 등 여러 장점을 가지고 있습니다. 자세히 설명해 드릴게요.

### 1. 배우기 쉽고 직관적인 문법 (Beginner-Fr



### finish_reason 확인

max_output_tokens에 도달하여 생성이 중단되면, 응답의 `finish_reason`이 `MAX_TOKENS`로 표시됩니다.
정상적으로 완료된 경우에는 `STOP`입니다.

이 정보를 활용하면 "응답이 잘렸는지" 프로그래밍적으로 판단할 수 있습니다.

아래 코드는 finish_reason을 확인합니다.

In [ ]:
# finish_reason 확인
prompt = "1부터 100까지의 소수를 모두 나열해줘"

# 충분한 토큰 → 정상 완료
resp_full = client.models.generate_content(
    model=MODEL,
    contents=prompt,
    config={"max_output_tokens": 500, "thinking_config": {"thinking_budget": 0}},
)
print(f"[max=500] finish_reason: {resp_full.candidates[0].finish_reason}")
print(f"  출력 토큰: {resp_full.usage_metadata.candidates_token_count}")
print(f"  응답 텍스트: {resp_full.text}")
print()

# 부족한 토큰 → 중간 절단
resp_short = client.models.generate_content(
    model=MODEL,
    contents=prompt,
    config={"max_output_tokens": 20, "thinking_config": {"thinking_budget": 0}},
)
print(f"[max=20]  finish_reason: {resp_short.candidates[0].finish_reason}")
print(f"  출력 토큰: {resp_short.usage_metadata.candidates_token_count}")
print(f"  잘린 텍스트: {resp_short.text}...")

[max=500] finish_reason: FinishReason.STOP
  출력 토큰: 108
  응답 텍스트: 1부터 100까지의 소수는 다음과 같습니다:

2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53, 59, 61, 67, 71, 73, 79, 83, 89, 97

[max=20]  finish_reason: FinishReason.MAX_TOKENS
  출력 토큰: 19
  잘린 텍스트: 1부터 100까지의 소수는 다음과 같습니다:

2, 3, ...


> 실무 팁: max_output_tokens는 비용 제어 수단으로도 유용합니다.
> 답변이 길어질 필요가 없는 분류/추출 작업에서는 50~100으로 제한하면 출력 토큰 비용을 절약할 수 있습니다.
> 단, 잘린 응답(finish_reason=MAX_TOKENS)이 사용자에게 노출되지 않도록 후처리가 필요합니다.

## 1.7 stop_sequences

**stop_sequences**는 모델이 특정 문자열을 생성하면 즉시 생성을 중단하는 파라미터입니다.
출력 포맷을 제어하거나, 불필요한 부분의 생성을 방지할 때 유용합니다.

- 최대 5개의 중단 시퀀스를 지정할 수 있음
- 중단 문자열 자체는 출력에 포함되지 않음

아래 코드는 stop_sequences로 생성을 중단하는 예시를 보여줍니다.

In [ ]:
# stop_sequences 기본 사용
prompt = "1부터 10까지 세어줘: 1, 2, 3, ... 이렇게"

# stop_sequences 없이
resp_no_stop = client.models.generate_content(
    model=MODEL,
    contents=prompt,
    config={"thinking_config": {"thinking_budget": 0}},
)
print(f"[stop 없음] {resp_no_stop.text}")
print(f"[stop reason] {resp_no_stop.candidates[0].finish_reason}")

# "7"에서 중단
resp_stop = client.models.generate_content(
    model=MODEL,
    contents=prompt,
    config={
        "stop_sequences": ["7"],
        "thinking_config": {"thinking_budget": 0},
    },
)
print(f"[stop='7'] {resp_stop.text}")
print(f"[stop reason] {resp_stop.candidates[0].finish_reason}")



[stop 없음] 1, 2, 3, 4, 5, 6, 7, 8, 9, 10
[stop reason] FinishReason.STOP
[stop='7'] 네, 1부터 10까지 세어드리겠습니다:

1, 2, 3, 4, 5, 6, 
[stop reason] FinishReason.STOP


핵심 포인트는 7이라는 토큰이 출력되기 직전에 생성이 멈추기 때문에, 7 자체는 응답에 포함되지 않는다는 점입니다. 그리고 finish_reason이 둘 다 동일하게 STOP으로 나오는데, 이는 Gemini API에서 정상 종료와 stop_sequences에 의한 종료를 같은 값으로 처리하기 때문입니다

### 복수 stop_sequences와 실용적 활용

여러 개의 중단 시퀀스를 동시에 지정할 수 있습니다.
실용적인 활용 예시:

- 구분자 기반 파싱: `"\n\n"`으로 첫 단락만 가져오기
- 리스트 제한: `"4."` 으로 상위 3개만 가져오기
- 코드 블록 끝: `"```"` 으로 코드 블록만 추출

아래 코드는 복수 stop_sequences를 활용하는 예시입니다.

In [ ]:
# 복수 stop_sequences 활용
# 리스트에서 상위 3개만 가져오기
prompt = "좋은 프로그래밍 습관 5가지를 번호를 매겨 알려줘"

resp = client.models.generate_content(
    model=MODEL,
    contents=prompt,
    config={
        "stop_sequences": ["3.", "3)"],  # 4번 항목이 시작되면 중단
        "thinking_config": {"thinking_budget": 0},
    },
)
print("[상위 3개만 출력]")
print(resp.text.strip())

print("\n" + "=" * 40)

# 첫 문장만 가져오기
prompt2 = "딥러닝에 대해 설명해줘"
resp2 = client.models.generate_content(
    model=MODEL,
    contents=prompt2,
    config={
        "stop_sequences": ["\n"],  # 첫 줄바꿈에서 중단
        "thinking_config": {"thinking_budget": 0},
    },
)
print("[첫 문장만]")
print(resp2.text.strip())

[상위 3개만 출력]
## 좋은 프로그래밍 습관 5가지

프로그래밍 실력을 향상시키고 유지 보수가 쉬운 코드를 작성하는 데 도움이 되는 좋은 프로그래밍 습관 5가지는 다음과 같습니다.

1.  **계획하고 설계하기:** 코드를 작성하기 전에 문제에 대해 생각하고 해결책을 계획하는 시간을 가지세요. 이것은 종이에 흐름도, 의사 코드, 또는 단순히 주석을 작성하는 것을 의미할 수 있습니다. 잘 계획된 접근 방식은 불필요한 반복과 버그를 줄이는 데 도움이 됩니다.

2.  **코드를 자주 테스트하기:** 작은 코드 조각을 작성할 때마다 테스트하여 의도한 대로 작동하는지 확인하세요. 이는 버그를 조기에 발견하고 디버깅을 더 쉽게 만듭니다. 나중에 더 복잡한 시스템에서 문제를 찾으려고 애쓰는 것보다 훨씬 효율적입니다.

[첫 문장만]
## 딥러닝 (Deep Learning) 이란?


## 1.8 seed

**seed**는 난수 생성의 시작점을 고정하여 재현 가능한 결과를 만드는 파라미터입니다.
같은 seed, 같은 프롬프트, 같은 파라미터를 사용하면 동일한 결과가 나올 가능성이 높아집니다.

- 실험 재현, A/B 테스트, 디버깅에 유용
- temperature > 0일 때도 결과를 고정할 수 있음

아래 코드는 seed를 사용한 재현성을 확인합니다.

In [ ]:
# seed를 사용한 재현성 확인
prompt = "재미있는 동물 이름을 하나 지어줘"

# seed 없이 (매번 다름)
print("=== seed 없이 (temperature=1.0) ===")
for i in range(3):
    resp = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config={"temperature": 1.0,
                "top_p" : 1.0,
                "top_k" : 70,
                "max_output_tokens" : 128,
                "thinking_config": {"thinking_budget": 0}
            },
    )
    print(f"  호출 {i+1}: {resp.text}")

print()

# seed 고정 (재현 가능)
print("=== seed=42 (temperature=1.0) ===")
for i in range(3):
    resp = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config={"temperature": 1.0,
                "top_p" : 1.0,
                "top_k" : 70,
                "max_output_tokens" : 128,
                "seed" : 42,
                "thinking_config": {"thinking_budget": 0}
            },
    )
    print(f"  호출 {i+1}: {resp.text}")

=== seed 없이 (temperature=1.0) ===
  호출 1: 네, 재미있는 동물 이름 하나를 지어드릴게요!

**"방귀대장 뿡뿡이"**

설명: 엉덩이에서 가끔 방귀를 뿡뿡 뀌는 특이한 습관을 가진 사랑스러운 작은 동물입니다. 냄새는 조금 나지만 워낙 귀엽고 재치가 넘쳐서 모두에게 인기를 독차지하는 마스코트예요! 😂

어떠세요, 재미있나요? 😊
  호출 2: 음... 재미있는 동물 이름이라. 한번 지어볼게요!

**별똥별 발바닥 젤리콩 (Starswift Pawprint Jellybean)**

어떤가요?

*   **별똥별:** 빠르고 반짝이는 느낌을 줘요.
*   **발바닥:** 귀여운 동물 발바닥은 상상만 해도 미소가 지어져요.
*   **젤리콩:** 작고 통통 튀면서 달콤한 느낌을 더해줘요.

왠지 밤하늘을 빠르게 뛰어다니는, 발바닥에
  호출 3: 재미있는 동물 이름으로 "무지개 엉덩이 너구리"는 어때요? 🦝

이름은 그 동물의 특징을 상상하는 데 도움이 될 거예요. 무지개 엉덩이를 가진 너구리는 알록달록한 색깔과 엉뚱한 매력을 갖고 있을 것 같습니다! ✨

=== seed=42 (temperature=1.0) ===
  호출 1: 재밌는 동물 이름! 흠... 

**"엉덩방아 깽깽이"** 어떠세요? 😂

왠지 깡총깡총 뛰다가 잘 넘어지는 귀여운 동물이 떠오르지 않나요?
  호출 2: 재밌는 동물 이름! 흠... 

**"엉덩방아 깽깽이"** 어떠세요? 😂

왠지 깡총깡총 뛰다가 잘 넘어지는 귀여운 동물이 떠오르지 않나요?
  호출 3: 재밌는 동물 이름! 흠... 

**"엉덩이 흔들 코끼리"** 어떠세요? 😂

왠지 코끼리가 신나서 엉덩이를 실룩거릴 것 같은 귀여운 느낌이죠?


### seed의 한계

seed를 사용해도 100% 동일한 결과가 보장되지는 않습니다.

- 모델 업데이트 시 결과가 달라질 수 있음
- 서버 인프라 변경에 영향을 받을 수 있음
- 같은 seed라도 프롬프트가 다르면 당연히 결과도 다름

> 핵심: seed는 "최선의 노력(best-effort)" 재현성입니다.
> 완벽한 재현이 필요하면 응답을 캐싱하는 것이 더 확실합니다.

아래 코드는 seed를 다르게 설정했을 때의 결과를 비교합니다.

> 파라미터 적용 우선순위 정리
> 1. **temperature**: 가장 먼저 조절. 용도에 따라 0.0~1.0 범위에서 결정
> 2. **top_p**: 기본값 유지(0.9~0.95)가 대부분 충분. 극단적 다양성이 필요할 때만 조절
> 3. **max_output_tokens**: 비용 제어와 응답 길이 제한에 사용
> 4. **stop_sequences**: 특정 포맷 제어가 필요할 때만 사용
> 5. **seed**: 실험 재현이 필요할 때만 설정
> 6. **top_k**: Gemini 전용. 일반적으로 기본값(40)을 유지

In [ ]:
# 서로 다른 seed, 높은 temperature → 서로 다른 결과
prompt = "오늘의 명언을 하나 만들어줘"

for seed_val in [1, 42, 100, 999]:
    resp = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config={
            "temperature": 1.0,
            "seed": seed_val,
            "thinking_config": {"thinking_budget": 0},
        },
    )
    print(f"[seed={seed_val:>3}] {resp.text}")

[seed=  1] 네, 오늘의 명언을 하나 만들어 드릴게요.

**"꿈을 향한 작은 한 걸음이, 결국 당신의 세상을 바꿀 위대한 여정의 시작이 될 것이다."**

이 명언이 당신에게 용기와 영감을 주기를 바랍니다. 😊
[seed= 42] 네, 오늘의 명언을 하나 만들어 드릴게요.

---

**오늘의 명언:**

**"어제의 후회에 갇히지 말고, 내일의 불안에 흔들리지 마라. 오직 오늘, 너의 진심으로 빛을 발하라."**

---

이 명언이 당신의 오늘 하루를 의미 있게 만들고, 긍정적인 에너지를 불어넣어 주길 바랍니다. 😊
[seed=100] ## 오늘의 명언:

**"바람에 흔들릴지언정, 뿌리 깊은 나무는 꺾이지 않는다. 흔들림 속에서 너의 단단함을 찾아라."**

---

**설명:**

이 명언은 우리 삶에 닥쳐오는 어려움이나 시련(바람) 속에서도, 우리 안에 있는 핵심 가치, 신념, 혹은 강점(뿌리 깊은 나무)을 잃지 않고 더욱 단단해질 수 있음을 말합니다. 흔들림은 피할 수 없지만, 그 안에서 좌절하기보다 자신을 돌아보고 더 강하게 만드는 기회로 삼으라는 메시지를 담고 있습니다.
[seed=999] 물론입니다! 오늘의 명언을 하나 만들어 드릴게요.

**"어제는 오늘의 스승이었고, 내일은 오늘이 빚어낼 찬란함이다."**

이 명언은 과거의 경험을 통해 배우고, 현재의 노력이 미래를 결정한다는 의미를 담고 있습니다. 😊


In [ ]:
# 서로 다른 seed, 낮은 temperature → 거의 같은 결과
prompt = "오늘의 명언을 하나 만들어줘"

for seed_val in [1, 42, 100, 999]:
    resp = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config={
            "temperature": 0.0,
            "seed": seed_val,
            "thinking_config": {"thinking_budget": 0},
        },
    )
    print(f"[seed={seed_val:>3}] {resp.text}")

[seed=  1] 네, 오늘의 명언을 하나 만들어 드릴게요.

**"어제의 후회는 오늘의 노력을 낳고, 오늘의 노력은 내일의 희망을 만든다."**

이 명언이 오늘 당신에게 작은 울림이 되기를 바랍니다. 😊
[seed= 42] 네, 오늘의 명언을 하나 만들어 드릴게요.

**"어제의 후회는 오늘의 노력을 낳고, 오늘의 노력은 내일의 희망을 만든다."**

이 명언이 오늘 당신에게 작은 울림이 되기를 바랍니다. 😊
[seed=100] 네, 오늘의 명언을 하나 만들어 드릴게요.

**"어제의 후회는 오늘의 노력을 낳고, 오늘의 노력은 내일의 희망을 만든다."**

이 명언이 오늘 하루를 살아가는 데 작은 힘이 되기를 바랍니다. 😊
[seed=999] 네, 오늘의 명언을 하나 만들어 드릴게요.

**"어제의 후회는 오늘의 노력을 낳고, 오늘의 노력은 내일의 희망을 만든다."**

이 명언이 오늘 하루를 살아가는 데 작은 힘이 되기를 바랍니다. 😊


In [ ]:
# 같은 seed에서, 온도가 점점 높아지는 경우 temperature → 서로 다른 결과
prompt = "오늘의 명언을 하나 만들어줘"

for temp in [0.0, 0.0, 0.0, 0.5, 0.5, 0.5, 1.0, 1.0, 1.0]:
    resp = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config={
            "temperature": temp,
            "seed": 42,
            "thinking_config": {"thinking_budget": 0},
        },
    )
    print(f"[seed=42] [temp={temp}] {resp.text}")

[seed=42] [temp=0.0] 네, 오늘의 명언을 하나 만들어 드릴게요.

**"어제의 후회는 오늘의 노력을 낳고, 오늘의 노력은 내일의 희망을 만든다."**

이 명언이 오늘 당신에게 작은 울림이 되기를 바랍니다. 😊
[seed=42] [temp=0.0] 네, 오늘의 명언을 하나 만들어 드릴게요.

**"어제의 후회는 오늘의 노력을 낳고, 오늘의 노력은 내일의 희망을 만든다."**

이 명언이 오늘 하루를 살아가는 데 작은 힘이 되기를 바랍니다. 😊
[seed=42] [temp=0.0] 네, 오늘의 명언을 하나 만들어 드릴게요.

**"어제의 후회는 오늘의 노력을 낳고, 오늘의 노력은 내일의 희망을 만든다."**

이 명언이 오늘 하루를 살아가는 데 작은 힘이 되기를 바랍니다. 😊
[seed=42] [temp=0.5] 네, 오늘의 명언을 하나 만들어 드릴게요.

**"어제의 후회는 오늘의 걸림돌이 되고, 내일의 기대는 오늘의 추진력이 된다."**

이 명언은 과거에 얽매이지 않고 현재에 집중하며 미래를 향해 나아가자는 메시지를 담고 있습니다. 마음에 드셨으면 좋겠네요! 😊
[seed=42] [temp=0.5] 네, 오늘의 명언을 하나 만들어 드릴게요.

**"어제의 후회는 오늘의 걸림돌이 되고, 내일의 기대는 오늘의 추진력이 된다."**

이 명언은 과거에 얽매이지 않고 현재에 집중하며 미래를 향해 나아가자는 메시지를 담고 있습니다. 마음에 드셨으면 좋겠네요! 😊
[seed=42] [temp=0.5] 네, 오늘의 명언을 하나 만들어 드릴게요.

**"어제의 후회는 오늘의 노력이 되고, 오늘의 노력은 내일의 희망이 된다."**

이 명언이 당신에게 작은 울림과 동기 부여가 되기를 바랍니다. 😊
[seed=42] [temp=1.0] 네, 오늘의 명언을 하나 만들어 드릴게요.

---

**오늘의 명언:**

**"어제의 후회는 오늘의 배움이 되고, 오늘의 노력은 내일의 희망이 된다."**

---

이 명언이 당신에게 작은 울림이나 동기가 되기를 바랍니다

## 1.9 google-genai config 총정리

지금까지 배운 모든 파라미터를 하나의 config에 모아서 사용할 수 있습니다.

아래 코드는 모든 생성 파라미터를 한 번에 설정하는 예시를 보여줍니다.

In [ ]:
# 모든 생성 파라미터를 한 번에 설정
response = client.models.generate_content(
    model=MODEL,
    contents="좋은 코딩 습관 3가지를 알려줘",
    config={
        "temperature": 0.3,
        "top_p": 0.95,
        "top_k": 40,
        "max_output_tokens": 200,
        "stop_sequences": ["4.", "4)"],
        "seed": 42,
        "thinking_config": {"thinking_budget": 0},
    },
)

print(response.text.strip())
print(f"\n--- 메타데이터 ---")
print(f"출력 토큰: {response.usage_metadata.candidates_token_count}")
print(f"finish_reason: {response.candidates[0].finish_reason}")

좋은 코딩 습관은 코드를 더 읽기 쉽고, 유지보수하기 쉬우며, 오류를 줄이는 데 도움이 됩니다. 다음은 3가지 핵심적인 좋은 코딩 습관입니다.

---

### 1. **명확하고 일관된 코딩 스타일 유지하기**

코딩 스타일은 코드의 가독성을 결정하는 가장 중요한 요소 중 하나입니다. 마치 글을 쓸 때 문법과 맞춤법을 지키는 것과 같습니다.

*   **변수, 함수, 클래스 이름 짓기:**
    *   **의미 있는 이름:** `x`, `tmp`와 같은 모호한 이름 대신 `userName`, `calculateTotalPrice`와 같이 변수나 함수가 무엇을 하는지 명확히 알 수 있는 이름을 사용하세요.
    *   **일관된 컨벤션:** `camelCase` (자바스크립트), `snake_case` (파

--- 메타데이터 ---
출력 토큰: 200
finish_reason: FinishReason.MAX_TOKENS


## 1.10 LangChain에서 파라미터 적용

LangChain의 `ChatGoogleGenerativeAI`에서도 동일한 파라미터를 사용할 수 있습니다.
생성자에서 직접 설정하거나, 호출 시 덮어쓸 수 있습니다.

아래 코드는 LangChain에서 생성 파라미터를 설정하는 방법을 보여줍니다.

참고로, langchain에서는 언어모델을 선언할 때 생성 config가 들어갑니다.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

# 생성자에서 파라미터 설정
llm = ChatGoogleGenerativeAI(
    model=MODEL,
    google_api_key=GEMINI_API_KEY,
    temperature=0.3,
    top_p=0.95,
    top_k=40,
    max_output_tokens=200,
    thinking_budget=0,
)

response = llm.invoke("파이썬의 장점을 한 문장으로 알려줘")
print(f"[LangChain temp=0.3]")
print(response.content)
print(f"\ntoken_usage: {response.usage_metadata}")

[LangChain temp=0.3]
파이썬은 **간결하고 읽기 쉬운 문법으로 빠르게 아이디어를 구현할 수 있는 강력한 언어**입니다.

token_usage: {'input_tokens': 13, 'output_tokens': 30, 'total_tokens': 43, 'input_token_details': {'cache_read': 0}}


### LCEL 체인에서 파라미터

LCEL 체인(`prompt | model | parser`)에서도 모델의 생성 파라미터가 그대로 적용됩니다.
체인을 여러 개 만들어 파라미터 조합별로 비교할 수도 있습니다.

아래 코드는 LCEL 체인에서 서로 다른 temperature를 사용하는 예시입니다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "한 문장으로 답변하세요."),
    ("human", "{question}"),
])
parser = StrOutputParser()

# 같은 체인 구조, 다른 temperature
for temp in [0.0, 0.7, 1.2]:
    model = ChatGoogleGenerativeAI(
        model=MODEL,
        google_api_key=GEMINI_API_KEY,
        temperature=temp,
        thinking_budget=0
    )
    chain = prompt_template | model.bind(stop=["해결"]) | parser
    result = chain.invoke({"question": "프로그래밍을 배워야 하는 이유는?"})
    print(f"[temp={temp}] {result}")

[temp=0.0] 문제 
[temp=0.7] 문제 
[temp=1.2] 프로그래밍은 문제 


> google-genai vs LangChain 파라미터 비교
>
> | 파라미터 | google-genai (config dict) | LangChain (생성자) |
> |---------|--------------------------|-------------------|
> | temperature | `"temperature": 0.3` | `temperature=0.3` |
> | top_p | `"top_p": 0.95` | `top_p=0.95` |
> | top_k | `"top_k": 40` | `top_k=40` |
> | max_output_tokens | `"max_output_tokens": 200` | `max_output_tokens=200` |
>
> LangChain은 여러 모델 제공자를 추상화하므로, stop_sequences처럼 제공자별 차이가 있는 파라미터는 `bind()` 또는 `model_kwargs`를 통해 전달합니다.

### 호출 시 파라미터 덮어쓰기

LangChain 모델은 생성자에서 기본 파라미터를 설정하고, `invoke()` 시 `config`로 일부를 덮어쓸 수 있습니다.
같은 모델 인스턴스를 다양한 상황에서 재사용할 때 유용합니다.

아래 코드는 `bind()` 메서드로 파라미터를 동적으로 변경하는 예시입니다.

In [ ]:
# bind()로 파라미터 덮어쓰기
base_model = ChatGoogleGenerativeAI(
    model=MODEL,
    google_api_key=GEMINI_API_KEY,
    temperature=0.7,
)

# 기본 설정으로 호출
resp1 = base_model.invoke("재미있는 사실 하나 알려줘")
print(f"[기본 temp=0.7] {resp1.content}")

# stop_sequences 추가 바인딩
strict_model = base_model.bind(stop=["\n"])
resp2 = strict_model.invoke("재미있는 사실 하나 알려줘")
print(f"[+stop='\\n'] {resp2.content}")

[기본 temp=0.7] 재미있는 사실 하나 알려드릴게요!

**나비는 발로 맛을 느껴요.**

꽃에 앉아 발로 즙의 맛을 본 다음, 맛있으면 주둥이를 펴서 즙을 빨아먹는답니다. 신기하죠? 😊
[+stop='\n'] 재미있는 사실 하나 알려드릴게요!


---

### 생각해보기

1. temperature를 0으로 설정했는데도 매번 미세하게 다른 결과가 나올 수 있을까요? 그렇다면 왜일까요?
2. 코드 생성에는 낮은 temperature, 창작에는 높은 temperature가 권장됩니다. 그렇다면 "챗봇을 하나 구성해줘"라는 요청에는 어떤 값이 적절할까요?
3. temperature와 top_p를 둘 다 최대로 올리면 어떤 일이 벌어질까요? 실제로 시도해보고 결과를 관찰해보세요.

---

# Part 2 — 실습

이론에서 배운 개념을 직접 코드로 작성해봅니다.
TODO 부분을 채워주세요. 막히면 정답 주석을 확인하세요.

### 실습 5-1: Temperature 응답 비교

temperature를 0.0과 1.5로 설정하여 동일한 프롬프트의 결과를 비교하세요.

**요구사항**
1. 프롬프트: "가을에 하면 좋은 일을 한 문장으로 추천해줘"
2. temperature=0.0과 temperature=1.5로 각각 호출
3. 두 결과를 출력하여 차이를 확인

In [ ]:
# TODO: temperature 0.0과 1.5의 결과를 비교하세요
prompt = "가을에 하면 좋은 일을 한 문장으로 추천해줘"

# ---------- 정답 ----------
for temp in [0.0, 1.5]:
    resp = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config={"temperature": temp, "thinking_config": {"thinking_budget": 0}},
    )
    print(f"[temp={temp}] {resp.text.strip()[:100]}")

print("TODO를 완성해주세요")

### 실습 5-2: Temperature 재현성 확인

temperature=0.0으로 같은 질문을 5번 호출하고, 결과가 동일한지 확인하세요.

**요구사항**
1. 프롬프트: "1+1은?"
2. temperature=0.0으로 5번 호출
3. 결과를 리스트에 저장
4. 모든 결과가 동일한지 `set()`으로 확인하여 출력

In [ ]:
# TODO: temperature=0 반복 호출 재현성 확인
prompt = "1+1은?"
results = []

# ---------- 정답 ----------
for i in range(5):
    resp = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config={"temperature": 0.0, "thinking_config": {"thinking_budget": 0}},
    )
    results.append(resp.text.strip())
    print(f"호출 {i+1}: {results[-1][:50]}")

unique = len(set(results))
print(f"\n고유 결과 수: {unique}/{len(results)}")
print(f"모든 결과 동일: {unique == 1}")

print("TODO를 완성해주세요")

### 실습 5-3: Top-p 실험

temperature를 고정(1.0)하고, top_p를 변화시켜 결과를 비교하세요.

**요구사항**
1. 프롬프트: "주말에 할 수 있는 취미를 하나 추천해줘"
2. temperature=1.0 고정
3. top_p를 0.1, 0.5, 0.9, 1.0으로 각각 호출
4. 결과를 비교 출력

In [ ]:
# TODO: top_p를 변화시켜 결과를 비교하세요
prompt = "주말에 할 수 있는 취미를 하나 추천해줘"
top_p_values = [0.1, 0.5, 0.9, 1.0]

# ---------- 정답 ----------
for tp in top_p_values:
    resp = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config={
            "temperature": 1.0,
            "top_p": tp,
            "max_output_tokens": 60,
            "thinking_config": {"thinking_budget": 0},
        },
    )
    text = resp.text.strip().split("\n")[0][:80]
    print(f"[top_p={tp}] {text}")

print("TODO를 완성해주세요")

### 실습 5-4: Temperature x Top-p 매트릭스

Temperature와 Top-p의 조합을 체계적으로 실험하세요.

**요구사항**
1. 프롬프트: "행복이란 무엇인가를 한 문장으로 정의해줘"
2. temperature: [0.0, 0.7, 1.5]
3. top_p: [0.5, 0.9, 1.0]
4. 3x3 = 9가지 조합을 모두 실행
5. 결과를 표 형태로 출력

In [ ]:
# TODO: Temperature x Top-p 매트릭스 실험
prompt = "행복이란 무엇인가를 한 문장으로 정의해줘"
temps = [0.0, 0.7, 1.5]
top_ps = [0.5, 0.9, 1.0]

# ---------- 정답 ----------
for temp in temps:
    for tp in top_ps:
        resp = client.models.generate_content(
            model=MODEL,
            contents=prompt,
            config={
                "temperature": temp,
                "top_p": tp,
                "max_output_tokens": 50,
                "thinking_config": {"thinking_budget": 0},
            },
        )
        text = resp.text.strip().replace("\n", " ")
        print(f"[temp={temp}, top_p={tp}] {text}")
    print()  # 그룹 구분

print("TODO를 완성해주세요")

### 실습 5-5: max_output_tokens 제어

max_output_tokens를 조절하여 출력 길이를 제어하고, finish_reason을 확인하세요.

**요구사항**
1. 프롬프트: "머신러닝의 종류와 각각의 특징을 설명해주세요"
2. max_output_tokens를 10, 50, 200으로 각각 호출
3. 각 결과의 실제 출력 토큰 수와 finish_reason을 출력

In [ ]:
# TODO: max_output_tokens 제어 + finish_reason 확인
prompt = "머신러닝의 종류와 각각의 특징을 설명해주세요"
max_tokens_list = [10, 50, 200]

# ---------- 정답 ----------
for max_t in max_tokens_list:
    resp = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config={"max_output_tokens": max_t, "thinking_config": {"thinking_budget": 0}},
    )
    u = resp.usage_metadata
    fr = resp.candidates[0].finish_reason
    print(f"[max={max_t:>3}] 출력: {u.candidates_token_count}토큰 | finish: {fr}")
    print(f"  {resp.text.strip()}")
    print()

print("TODO를 완성해주세요")

### 실습 5-6: stop_sequences 활용

stop_sequences를 사용하여 원하는 지점에서 생성을 중단하세요.

**요구사항**
1. 프롬프트: "대한민국의 대도시 5개를 번호를 매겨 나열해줘"
2. stop_sequences로 3번 항목까지만 출력되게 설정
3. stop_sequences 있는 결과와 없는 결과를 비교 출력

In [ ]:
# TODO: stop_sequences로 생성 중단 제어
prompt = "대한민국의 대도시 5개를 번호를 매겨 나열해줘"

# ---------- 정답 ----------
# stop 없이
resp_full = client.models.generate_content(
    model=MODEL,
    contents=prompt,
    config={"thinking_config": {"thinking_budget": 0}},
)
print("[stop 없음]")
print(resp_full.text.strip())

print("\n" + "=" * 30)

# 4번부터 차단
resp_stop = client.models.generate_content(
    model=MODEL,
    contents=prompt,
    config={
        "stop_sequences": ["4.", "4)"],
        "thinking_config": {"thinking_budget": 0},
    },
)
print("[stop='4.'/'4)']")
print(resp_stop.text.strip())

print("TODO를 완성해주세요")

### 실습 5-7: LangChain에서 파라미터 적용

LangChain의 LCEL 체인에서 생성 파라미터를 적용하세요.

**요구사항**
1. ChatGoogleGenerativeAI 모델 생성 (temperature=0.3, max_output_tokens=100)
2. ChatPromptTemplate: system="한국어로 간결하게 답변하세요", human="{topic}에 대해 알려줘"
3. StrOutputParser로 체인 구성
4. topic="양자컴퓨팅"으로 호출
5. 결과 출력

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# TODO: LangChain LCEL 체인 구성 + 파라미터 적용

# ---------- 정답 ----------
model = ChatGoogleGenerativeAI(
    model=MODEL,
    google_api_key=GEMINI_API_KEY,
    temperature=0.3,
    max_output_tokens=100,
    thinking_budget=20
)

prompt_tpl = ChatPromptTemplate.from_messages([
    ("system", "한국어로 간결하게 답변하세요"),
    ("human", "{topic}에 대해 알려줘"),
])

chain = prompt_tpl | model | StrOutputParser()
result = chain.invoke({"topic": "양자컴퓨팅"})
print(result)

print("TODO를 완성해주세요")

---

### 생각해보기

1. 실습 5-4의 매트릭스에서 temp=0.0일 때 top_p를 바꿔도 결과가 같았나요? 왜 그런 걸까요?
3. seed를 사용하면 실험 재현이 가능하지만 100% 보장은 안 됩니다. 완벽한 재현성이 필요한 경우 어떤 전략을 쓸 수 있을까요?

---

# Part 3 — 챌린지

이론과 실습에서 배운 내용을 응용하여 스스로 설계하고 구현합니다.
정답이 제공되지 않으며, 강사와 함께 진행합니다.

### 챌린지 5-1: 용도별 최적 파라미터 가이드 도출 (난이도: ★☆☆)

> 이 챌린지는 정답이 제공되지 않습니다. 강사와 함께 진행하세요.

2가지 서로 다른 용도에 대해 최적의 파라미터 조합을 실험적으로 도출하세요.

**용도 목록**
1. 영어-한국어 번역
2. 뉴스 기사 요약

**과제**
- 각 용도에 맞는 프롬프트를 작성
- 가장 좋은 조합을 선택하고, 그 이유를 설명
- 최종 가이드를 표 형태로 정리

**힌트**
- 정확성이 중요한 작업 vs 다양성이 중요한 작업을 먼저 구분하세요
- max_output_tokens도 용도에 맞게 조절해보세요
- 같은 조합으로 3번 호출하여 일관성도 확인하세요

In [ ]:
# ── 1단계: 용도별 프롬프트 + 파라미터 실험 ──
tasks = {
    "번역": {
        "prompt": "Translate the following English sentence into natural Korean:\n'The rapid advancement of artificial intelligence is reshaping industries worldwide.'",
        "candidates": [
            {"temperature": 0.0, "top_p": 1.0, "max_output_tokens": 100},
            {"temperature": 0.3, "top_p": 0.95, "max_output_tokens": 100},
            {"temperature": 0.7, "top_p": 0.9, "max_output_tokens": 100},
        ],
    },
    "뉴스 요약": {
        "prompt": (
            "다음 뉴스 기사를 3문장으로 요약하세요.\n\n"
            "인공지능 기술이 의료 분야에서 빠르게 확산되고 있다. "
            "특히 영상 진단 보조, 신약 개발, 환자 모니터링 등에서 활용 사례가 늘고 있으며, "
            "주요 병원들은 AI 기반 시스템을 도입하여 진단 정확도를 높이고 있다. "
            "다만 의료 데이터 프라이버시와 AI 판단의 책임 소재에 대한 논의는 여전히 진행 중이다."
        ),
        "candidates": [
            {"temperature": 0.0, "top_p": 1.0, "max_output_tokens": 200},
            {"temperature": 0.3, "top_p": 0.95, "max_output_tokens": 200},
            {"temperature": 0.7, "top_p": 0.9, "max_output_tokens": 200},
        ],
    },
}

for task_name, task in tasks.items():
    print(f"{'=' * 50}")
    print(f"[{task_name}]")
    print(f"{'=' * 50}")
    for params in task["candidates"]:
        config = {**params, "thinking_config": {"thinking_budget": 0}}
        # 3회 반복으로 일관성 확인
        results = []
        for _ in range(3):
            resp = client.models.generate_content(
                model=MODEL, contents=task["prompt"], config=config,
            )
            results.append(resp.text.strip().replace("\n", " ")[:120])

        unique_count = len(set(results))
        print(f"\n  temp={params['temperature']}, top_p={params['top_p']}")
        print(f"  응답 예시: {results[0]}")
        print(f"  3회 중 고유 결과 수: {unique_count}/3")
    print()


---

### 생각해보기

1. 파라미터 가이드를 만들었는데, 모델이 업데이트되면 이 가이드도 다시 만들어야 할까요? 모델 버전에 따른 파라미터 민감도를 어떻게 관리할 수 있을까요?
2. 용도에 따라 최적 파라미터가 다른 것은, 결국 "정확성 vs 다양성" 트레이드오프입니다. 이 둘이 동시에 중요한 용도(예: 교육용 설명 생성)에는 어떤 전략이 좋을까요?
3. 실제 서비스에서 생성 파라미터를 사용자에게 노출해야 할까요, 아니면 백엔드에서 용도별로 고정해야 할까요? 각각의 장단점을 비교해보세요.